# Scalar Chebyshev2 vs Trapezoid: Fixed Sample Counts

This is a smaller companion to `scalar_quadrature_comparison.ipynb`. Each subplot fixes the number of samples `N`; the y-axis is the Chebyshev node count `m = 2..10`. The light grey dashed line marks `sqrt(N)`.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / "python").exists():
    repo_root = repo_root.parent
python_dir = repo_root / "python"
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

import imuFactors.scalar_quadrature as scalar_quadrature

plt.rcParams.update({"figure.dpi": 120})

In [ ]:
INTERVAL = (0.0, 1.0)
FUNCTIONS = [
    scalar_quadrature.ScalarFunction(
        name="tanh(t)",
        value=lambda t: np.tanh(t),
        antiderivative=lambda t: np.log(np.cosh(t)),
    ),
    scalar_quadrature.ScalarFunction(
        name="sin(2*pi*t)",
        value=lambda t: np.sin(2.0 * np.pi * t),
        antiderivative=lambda t: (1.0 - np.cos(2.0 * np.pi * t)) / (2.0 * np.pi),
    ),
    scalar_quadrature.ScalarFunction(
        name="pi*t^(31/4)",
        value=lambda t: np.pi * t ** (31.0 / 4.0),
        antiderivative=lambda t: np.pi * t ** (35.0 / 4.0) / (35.0 / 4.0),
    ),
]

SAMPLE_COUNTS = [10, 20, 30, 40, 50]
CHEBYSHEV_NODES = np.arange(2, 11)
NOISE_FRACTIONS = np.array([
    0.0, 0.025, 0.05, 0.06, 0.075, 0.10,
    0.12, 0.15, 0.17, 0.20, 0.225,
])
NUM_SEEDS = 100
RANDOM_SEED = 20260523
EVALUATION_COUNT = 151

In [ ]:
node_ranges = {
    sample_count: CHEBYSHEV_NODES
    for sample_count in SAMPLE_COUNTS
}
pd.DataFrame(
    {
        "N": list(node_ranges),
        "m_min": [values[0] for values in node_ranges.values()],
        "m_max": [values[-1] for values in node_ranges.values()],
        "sqrt_N": [np.sqrt(sample_count) for sample_count in node_ranges],
        "num_m": [len(values) for values in node_ranges.values()],
    }
)

In [ ]:
runs = []
for sample_count, node_counts in node_ranges.items():
    runs.append(
        scalar_quadrature.run_scalar_monte_carlo(
            FUNCTIONS,
            sample_counts=[sample_count],
            chebyshev_node_counts=node_counts,
            noise_fractions=NOISE_FRACTIONS,
            num_seeds=NUM_SEEDS,
            seed=RANDOM_SEED,
            interval=INTERVAL,
            evaluation_count=EVALUATION_COUNT,
        )
    )

method_metrics = pd.concat(
    [run.method_metrics for run in runs], ignore_index=True
)
comparisons = pd.concat(
    [run.comparisons for run in runs], ignore_index=True
)
comparisons.head()

In [ ]:
for function in FUNCTIONS:
    fig = scalar_quadrature.plot_fixed_sample_comparison(
        comparisons,
        function_name=function.name,
        selected_sample_counts=SAMPLE_COUNTS,
        show_sqrt_sample_count=True,
    )
    display(fig)
    plt.close(fig)

In [ ]:
summary = (
    comparisons.groupby(["function", "sample_count"])[["end_error", "rmse_error", "max_error"]]
    .median()
    .round(6)
)
summary

## Decision-oriented Plotly views

These views use the same comparison dataframe as the heatmaps above. Positive advantage is `trapezoid error - Chebyshev2 error`, so values above zero favor Chebyshev2. The diamond marks the best median RMSE `m`; the light grey dashed line is `sqrt(N)`. The table reports ideal `m` by metric plus a rank-based robust `m`.

In [ ]:
for function in FUNCTIONS:
    display(
        scalar_quadrature.plot_advantage_curves_by_sample_count(
            comparisons,
            function.name,
            selected_sample_counts=SAMPLE_COUNTS,
            metric="rmse_error",y_range_min_m=3
        )
    )
    display(
        scalar_quadrature.plot_robust_m_table(
            comparisons,
            function.name,
            selected_sample_counts=SAMPLE_COUNTS,
        )
    )